# Unsloshをつかったgptoss20bのSFT

### ライブラリロード

In [1]:
from unsloth import FastLanguageModel
import torch

# 128GBのメモリを活かし、最高精度（BF16）かつ大容量の文脈でロード
max_seq_length = 4096 # 200ページの情報を効率よくパッキングするために4096以上を推奨
dtype = torch.bfloat16 # DGX Spark/BlackwellならBF16が最適
load_in_4bit = False   # メモリに余裕があるため、量子化なしで精度を優先

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b", # Hugging Face上の正確なリポジトリ名を確認してください
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # device_map = "auto", # Unslothが自動でGPUに割り当てます
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.8: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260322. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards: 100%|██| 3/3 [01:29<00:00, 29.77s/it]


### PEFT/LoRAの設定

In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, 
    # 特定の層だけでなく、すべての線形層（全結合層）を対象にします
    target_modules = "all-linear", 
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

Unsloth: Detected MoE model with num_experts = 32 and target_modules = '(?:.*?(?:vision|image|visual|patch|language|text).*?(?:self_attn|attention|attn|mlp|feed_forward|ffn|dense).*?(?:q_proj|k_proj|v_proj|o_proj).*?)|(?:\\bmodel\\.layers\\.[\\d]{1,}\\.(?:self_attn|attention|attn|mlp|feed_forward|ffn|dense)\\.(?:(?:q_proj|k_proj|v_proj|o_proj)))'. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']
Unsloth: PEFT set target_parameters but found no matching parameters.
This is expected for MoE models - Unsloth handles MoE expert LoRA targeting separately.
Unsloth: Making `model.base_model.model.model` require gradients


ステップ2：作成したJSONLデータの読み込み

In [3]:
from datasets import load_dataset

# ファイルパスを指定して読み込み
dataset = load_dataset("json", data_files={"train": "cpt_input.jsonl"}, split="train")

# データの形式を確認（"text" カラムがあることを想定）
print(dataset[0])

{'text': '# 障害程度別対象事業一覧表 (○対象・△一部対象)\n\n以下は、障害程度別の対象となる事業の一覧です。\n\n- **東京メトロ旅客運賃の割引**（本文頁：120）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **私鉄旅客運賃の割引**（本文頁：120）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **タクシー運賃の割引**（本文頁：120）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **航空運賃の割引**（本文頁：121）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **旅客船運賃の割引**（本文頁：121）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **テレビ受信料の減免**（本文頁：121）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが一部対象となります。所得制限があります。\n- **水道・下水道料金の減免**（本文頁：122）詳細は本文を参照してください。所得制限があります。\n- **携帯電話使用料等の割引**（本文頁：123）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **郵便料金・ゆうパック運賃等の減免**（本文頁：123）詳細は本文を参照してください。所得制限はありません。\n- **通常郵便葉書（青い鳥郵便葉書）の無料配布**（本文頁：124）視覚障害等級 1、2、3 および聴覚又は平衡感覚機能障害等級 2 が対象となります。所得制限はありません。\n\n### **税の軽減**\n\n税の軽減に関する事業は以下の通りです。\n\n- **所得税・住民税の障害者控除**（本文頁：125）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありま

### 学習

### 学習用メソッドの定義

In [4]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bf16_supported
import time

def train(args):
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = dataset,
        dataset_text_field = "text",
        max_seq_length = 4096,
        packing = True, 
        args = args,
    )
    
    start_time = time.time()
    print(f"学習を開始します... (開始時刻: {time.ctime(start_time)})")
    # 学習開始！
    trainer_stats = trainer.train()
    # --- 学習終了後の時刻を記録 ---
    end_time = time.time()
    total_time = end_time - start_time
    
    # 秒を「分:秒」の形式に変換
    minutes = int(total_time // 60)
    seconds = int(total_time % 60)

    print("-" * 30)
    print(f"学習が完了しました！ (終了時刻: {time.ctime(end_time)})")
    print(f"合計学習時間: {minutes}分 {seconds}秒")
    print(f"1ステップあたりの平均時間: {total_time / trainer.args.max_steps:.2f}秒")
    print("-" * 30)
    

#### 学習用パラメータ

In [5]:
args = TrainingArguments(
        per_device_train_batch_size = 8, 
        gradient_accumulation_steps = 4, 
        warmup_steps = 50,
        max_steps = 200, 
        learning_rate = 4e-4, 
        lr_scheduler_type = "cosine", 
        bf16 = True, # DGX Spark/Blackwellなら必須の設定
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        output_dir = "outputs_gpt_oss_v3",
        save_steps = 100, # 途中で保存しておく
    )

#### 学習実行

In [7]:
train(args)

学習を開始します... (開始時刻: Sun Mar 22 23:22:13 2026)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 218 | Num Epochs = 29 | Total steps = 200
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 15,925,248 of 1,820,384,832 (0.87% trained)
[unsloth_zoo.log|WARNING]Unsloth: GPT-OSS MoE training path: MXFP4 + triton_kernels. LoRA=False, experts=32. Tip: Increase batch_size for better GPU utilization.
Autotune Choices Stats:
{"num_choices": 23, "num_triton_choices": 23, "best_kernel": "triton_flex_attention_backward_7", "best_kernel_desc": "BLOCKS_ARE_CONTIGUOUS=False, BLOCK_M1=64, BLOCK_M2=64, BLOCK_N1=64, BLOCK_N2=64, FLOAT32_PRECISION=\"'tf32'\", GQA_SHARED_HEADS=8, HAS_FULL_BLOCKS=True, IS_DIVISIBLE=True, OUTPUT_LOGSUMEXP=True, OUTPUT_MAX=False, PRESCALE_QK=False, QK_HEAD_DIM=64, QK_HEAD_DIM_ROUNDED=64, ROWS_GUARANTEED_SAFE=False, SAFE_HEAD_DIM=True, SM_SCAL

Step,Training Loss
1,3.044800
2,3.046000
3,3.078000
4,3.046200
5,2.894800
6,2.722200
7,2.626000
8,2.429400
9,2.027700
10,1.784500


Autotune Choices Stats:
{"num_choices": 23, "num_triton_choices": 23, "best_kernel": "triton_flex_attention_backward_53", "best_kernel_desc": "BLOCKS_ARE_CONTIGUOUS=False, BLOCK_M1=64, BLOCK_M2=64, BLOCK_N1=64, BLOCK_N2=64, FLOAT32_PRECISION=\"'tf32'\", GQA_SHARED_HEADS=8, HAS_FULL_BLOCKS=True, IS_DIVISIBLE=True, OUTPUT_LOGSUMEXP=True, OUTPUT_MAX=False, PRESCALE_QK=False, QK_HEAD_DIM=64, QK_HEAD_DIM_ROUNDED=64, ROWS_GUARANTEED_SAFE=False, SAFE_HEAD_DIM=True, SM_SCALE=0.125, SPARSE_KV_BLOCK_SIZE=128, SPARSE_Q_BLOCK_SIZE=128, V_HEAD_DIM=64, V_HEAD_DIM_ROUNDED=64, WRITE_DQ=True, num_stages=3, num_warps=4", "best_time": 3.7541439533233643, "best_triton_pos": 0}
Autotune Choices Stats:
{"num_choices": 23, "num_triton_choices": 23, "best_kernel": "triton_flex_attention_backward_78", "best_kernel_desc": "BLOCKS_ARE_CONTIGUOUS=False, BLOCK_M1=64, BLOCK_M2=64, BLOCK_N1=64, BLOCK_N2=64, FLOAT32_PRECISION=\"'tf32'\", GQA_SHARED_HEADS=8, HAS_FULL_BLOCKS=True, IS_DIVISIBLE=True, OUTPUT_LOGSUMEXP=Tr

------------------------------
学習が完了しました！ (終了時刻: Mon Mar 23 00:38:51 2026)
合計学習時間: 76分 38秒
1ステップあたりの平均時間: 22.99秒
------------------------------


### 推論テスト

In [11]:
# プロンプトのテンプレートを定義
prompt_style = """以下は、障害者福祉の手続きに関するガイドラインに基づいた回答です。
質問に対して、正確な日本語で回答してください。

### 質問:
{}

### 回答:
"""

# 質問をテンプレートに埋め込む
input_text = prompt_style.format("身体障害者手帳の申請に必要なものと、電車を利用する際の手続きについて教えてください。")

inputs = tokenizer([input_text], return_tensors = "pt").to("cuda")

# 生成（パラメータを微調整：より確実な言葉を選ばせる）
outputs = model.generate(
    **inputs, 
    max_new_tokens = 300,
    temperature = 0.1,          # 低くして「迷い」を消す
    repetition_penalty = 1.2,   # 同じ言葉のループを防ぐ
    top_p = 0.9,
    use_cache = True
)

print(tokenizer.batch_decode(outputs, skip_special_tokens = True)[0])

以下は、障害者福祉の手続きに関するガイドラインに基づいた回答です。
質問に対して、正確な日本語で回答してください。

### 質問:
身体障害者手帳の申請に必要なものと、電車を利用する際の手続きについて教えてください。

### 回答:
東京メトロ旅客運賃の割引（120 ページ）は所得制限なく設用されておい。視覚障害 1級から6 級まで持つ方があり、それ自動障害 2 級から4 級まる等級も有べます。この制度では、すべても税金の軽減や郵便料金・ゆうパック運賄などの削除の対象となります。また、水道・下水道料金の減免については詳細な条件が本文表中記されていますので、具体的なおよび詈別貯�ーションの利上子助成には同様に全所無罪です。

携帯電話使用料等シんみたし工業賃の割引（123 ページ）も所得制限なしに適用されています。同様に、通常住宅設備改善費の非課税区隔があります。ただし、この場合「都営船」として参照していますため、視覚障害者手帳の種類によって異なる程度の支援内容がありば存在します。

## 税の軽減
個住民事財源の減額については、所得制限はありません。テレビ受信料の減免措置は所得制限なく、視覚障害 1-3 級、および自動障害 2 級


# モデルの保存

In [13]:
model.save_pretrained("gpt_oss_disabled_support_v1")
tokenizer.save_pretrained("gpt_oss_disabled_support_v1")


('gpt_oss_disabled_support_v1/tokenizer_config.json',
 'gpt_oss_disabled_support_v1/special_tokens_map.json',
 'gpt_oss_disabled_support_v1/chat_template.jinja',
 'gpt_oss_disabled_support_v1/tokenizer.json')